# Multimodal Foundation Model Training

This notebook demonstrates end-to-end training of multimodal vision-language models using our framework. We'll train CLIP with LoRA and compare different configurations.

## Objectives
1. Set up training configuration
2. Initialize CLIP model with LoRA
3. Create data loaders and preprocessing
4. Train model with distributed setup
5. Monitor training progress
6. Evaluate model performance

In [ ]:
import sys
sys.path.append('../')

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Training Configuration Setup

In [ ]:
# Training configuration
from src.training.distributed_trainer import TrainingConfig

config = TrainingConfig(
    # Model configuration
    model_name="openai/clip-vit-base-patch32",
    
    # Training hyperparameters
    batch_size=16,  # Adjusted for notebook training
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=2,  # Short training for demo
    
    # Distributed training (disabled for notebook)
    use_fsdp=False,
    use_deepspeed=False,
    
    # Mixed precision
    bf16=torch.cuda.is_available(),
    fp16=False,
    
    # Paths
    output_dir="./training_outputs",
    
    # Logging
    logging_steps=10,
    save_steps=100,
    eval_steps=50,
    
    # MLflow
    use_mlflow=False,  # Disable for notebook
    experiment_name="notebook-training",
    
    # Reproducibility
    seed=42
)

print("Training Configuration:")
for key, value in config.to_dict().items():
    print(f"  {key}: {value}")

# Create output directory
Path(config.output_dir).mkdir(parents=True, exist_ok=True)

## 2. Model Initialization

In [ ]:
from src.models.clip_lora import CLIPLoRAModel

# Initialize CLIP model with LoRA
model = CLIPLoRAModel(
    model_name=config.model_name,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    device=str(device)
)

print("Model initialized successfully!")
model.print_trainable_parameters()

# Display model architecture summary
total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"\nModel Summary:")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Trainable Ratio: {100 * trainable_params / total_params:.2f}%")

## 3. Data Loading and Preprocessing

In [ ]:
from src.data.dataloader import create_dataloaders
from datasets import load_dataset
import torch.utils.data as data
from PIL import Image

# Create a simple dataset for training demonstration
class SimpleMultimodalDataset(data.Dataset):
    def __init__(self, processor, num_samples=100):
        self.processor = processor
        self.num_samples = num_samples
        
        # Generate synthetic data for demonstration
        self.data = []
        captions = [
            "A dog playing in the park",
            "A cat sitting on a chair",
            "A bird flying in the sky",
            "A car driving on the road",
            "A person walking on the street",
            "A tree in the forest",
            "A house with a garden",
            "A mountain landscape",
            "A beach with waves",
            "A city skyline at night"
        ]
        
        for i in range(num_samples):
            # Create synthetic image
            color = (np.random.randint(50, 200), np.random.randint(50, 200), np.random.randint(50, 200))
            image = Image.new('RGB', (224, 224), color=color)
            
            # Select caption
            caption = captions[i % len(captions)]
            
            self.data.append({
                'image': image,
                'caption': caption
            })
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Process with CLIP processor
        inputs = self.processor(
            images=item['image'],
            text=item['caption'],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77  # CLIP max length
        )
        
        # Remove batch dimension and add return_loss flag
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'return_loss': True
        }

# Create datasets
train_dataset = SimpleMultimodalDataset(model.processor, num_samples=200)
val_dataset = SimpleMultimodalDataset(model.processor, num_samples=50)

# Create data loaders
train_dataloader = data.DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

val_dataloader = data.DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True if device.type == 'cuda' else False
)

print(f"Created data loaders:")
print(f"  Train batches: {len(train_dataloader)}")
print(f"  Validation batches: {len(val_dataloader)}")

# Test data loading
sample_batch = next(iter(train_dataloader))
print(f"\nSample batch shapes:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: {value.shape}")
    else:
        print(f"  {key}: {value}")

## 4. Training Setup and Execution

In [ ]:
from src.training.accelerate_trainer import AccelerateTrainer
from src.models.clip_lora import CLIPContrastiveLoss
import mlflow

# Disable MLflow for notebook training
os.environ['MLFLOW_TRACKING_URI'] = ''

# Initialize trainer
try:
    trainer = AccelerateTrainer(
        model=model.model,  # Pass the underlying model
        config=config,
        train_dataloader=train_dataloader,
        eval_dataloader=val_dataloader,
        compute_metrics=None  # We'll implement custom metrics
    )
    print("Trainer initialized successfully!")
except Exception as e:
    print(f"Error initializing trainer: {e}")
    print("Falling back to simple training loop...")
    trainer = None

In [ ]:
# Simple training loop implementation if accelerate fails
if trainer is None:
    print("Running simple training loop...")
    
    # Set up optimizer and loss
    optimizer = torch.optim.AdamW(
        model.model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )
    
    criterion = CLIPContrastiveLoss(temperature=0.07)
    
    # Training history
    training_history = []
    
    model.model.train()
    
    print("Starting training...")
    
    for epoch in range(config.num_train_epochs):
        epoch_loss = 0
        num_batches = 0
        
        for batch_idx, batch in enumerate(train_dataloader):
            # Move batch to device
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in batch.items()}
            
            # Forward pass
            outputs = model.model(**batch)
            loss = outputs.loss if hasattr(outputs, 'loss') else outputs
            
            # Backward pass
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            epoch_loss += loss.item()
            num_batches += 1
            
            if batch_idx % config.logging_steps == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item():.4f}")
        
        avg_loss = epoch_loss / num_batches
        training_history.append({
            'epoch': epoch,
            'train_loss': avg_loss
        })
        
        print(f"Epoch {epoch} completed. Average loss: {avg_loss:.4f}")
        
        # Simple validation
        if val_dataloader is not None:
            model.model.eval()
            val_loss = 0
            val_batches = 0
            
            with torch.no_grad():
                for val_batch in val_dataloader:
                    val_batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                               for k, v in val_batch.items()}
                    
                    val_outputs = model.model(**val_batch)
                    val_loss += (val_outputs.loss if hasattr(val_outputs, 'loss') else val_outputs).item()
                    val_batches += 1
            
            avg_val_loss = val_loss / val_batches
            training_history[-1]['val_loss'] = avg_val_loss
            print(f"Validation loss: {avg_val_loss:.4f}")
            
            model.model.train()
    
    print("Training completed!")
    
else:
    # Use accelerate trainer
    print("Starting training with Accelerate...")
    results = trainer.train()
    training_history = results.get('training_history', [])
    print("Training completed!")

## 5. Training Progress Visualization

In [ ]:
# Visualize training progress
if training_history:
    import pandas as pd
    
    df = pd.DataFrame(training_history)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Training loss
    axes[0].plot(df['epoch'], df['train_loss'], 'b-', label='Training Loss', marker='o')
    if 'val_loss' in df.columns:
        axes[0].plot(df['epoch'], df['val_loss'], 'r-', label='Validation Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Progress')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss improvement
    if len(df) > 1:
        loss_improvement = df['train_loss'].iloc[0] - df['train_loss'].iloc[-1]
        improvement_pct = (loss_improvement / df['train_loss'].iloc[0]) * 100
        
        axes[1].bar(['Initial Loss', 'Final Loss'], 
                   [df['train_loss'].iloc[0], df['train_loss'].iloc[-1]],
                   color=['red', 'green'], alpha=0.7)
        axes[1].set_ylabel('Loss')
        axes[1].set_title(f'Loss Improvement: {improvement_pct:.1f}%')
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{config.output_dir}/training_progress.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print final statistics
    print("\nTraining Statistics:")
    print(f"Initial training loss: {df['train_loss'].iloc[0]:.4f}")
    print(f"Final training loss: {df['train_loss'].iloc[-1]:.4f}")
    if 'val_loss' in df.columns:
        print(f"Final validation loss: {df['val_loss'].iloc[-1]:.4f}")
    
    if len(df) > 1:
        print(f"Total improvement: {improvement_pct:.1f}%")
else:
    print("No training history available for visualization")

## 6. Model Evaluation

In [ ]:
# Evaluate trained model
from src.evaluation.metrics import VisionLanguageMetrics

print("Evaluating trained model...")

model.model.eval()

# Test text and image encoding
test_texts = [
    "A dog playing in the park",
    "A cat sitting on a chair",
    "A bird flying in the sky",
    "A car driving on the road"
]

# Create test images (synthetic for demo)
test_images = []
for i, text in enumerate(test_texts):
    color = (100 + i * 30, 150, 200 - i * 20)
    image = Image.new('RGB', (224, 224), color=color)
    test_images.append(image)

# Test text encoding
try:
    text_embeddings = model.encode_text(test_texts)
    print(f"Text embeddings shape: {text_embeddings.shape}")
    print(f"Text embeddings norm (mean): {torch.norm(text_embeddings, dim=-1).mean().item():.4f}")
except Exception as e:
    print(f"Error encoding text: {e}")

# Test image encoding
try:
    image_embeddings = model.encode_image(test_images)
    print(f"Image embeddings shape: {image_embeddings.shape}")
    print(f"Image embeddings norm (mean): {torch.norm(image_embeddings, dim=-1).mean().item():.4f}")
except Exception as e:
    print(f"Error encoding images: {e}")

# Test similarity computation
try:
    similarities = model.compute_similarity(test_texts, test_images)
    print(f"\nText-Image Similarity Matrix:")
    print(similarities.cpu().numpy())
    
    # Check if diagonal is highest (correct matching)
    diagonal_mean = torch.diag(similarities).mean().item()
    off_diagonal_mean = (similarities.sum() - torch.diag(similarities).sum()) / (similarities.numel() - similarities.size(0))
    
    print(f"\nDiagonal similarity (correct matches): {diagonal_mean:.4f}")
    print(f"Off-diagonal similarity (incorrect matches): {off_diagonal_mean.item():.4f}")
    print(f"Alignment score: {diagonal_mean - off_diagonal_mean.item():.4f}")
    
except Exception as e:
    print(f"Error computing similarities: {e}")

## 7. Model Comparison and Analysis

In [ ]:
# Compare different LoRA configurations
print("Comparing different LoRA configurations...")

lora_configs = [
    {'r': 8, 'alpha': 16, 'name': 'LoRA-8'},
    {'r': 16, 'alpha': 32, 'name': 'LoRA-16'},
    {'r': 32, 'alpha': 64, 'name': 'LoRA-32'}
]

comparison_results = []

for config_item in lora_configs:
    try:
        # Create model with different LoRA config
        temp_model = CLIPLoRAModel(
            model_name="openai/clip-vit-base-patch32",
            lora_r=config_item['r'],
            lora_alpha=config_item['alpha'],
            device=str(device)
        )
        
        # Count parameters
        total_params = sum(p.numel() for p in temp_model.model.parameters())
        trainable_params = sum(p.numel() for p in temp_model.model.parameters() if p.requires_grad)
        
        comparison_results.append({
            'config': config_item['name'],
            'lora_r': config_item['r'],
            'lora_alpha': config_item['alpha'],
            'total_params': total_params,
            'trainable_params': trainable_params,
            'trainable_ratio': trainable_params / total_params
        })
        
        del temp_model  # Clean up memory
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"Error with {config_item['name']}: {e}")

# Display comparison
if comparison_results:
    df_comparison = pd.DataFrame(comparison_results)
    
    print("\nLoRA Configuration Comparison:")
    print(df_comparison[['config', 'trainable_params', 'trainable_ratio']].to_string(index=False))
    
    # Visualize parameter efficiency
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Trainable parameters
    axes[0].bar(df_comparison['config'], df_comparison['trainable_params'], alpha=0.7)
    axes[0].set_title('Trainable Parameters by LoRA Configuration')
    axes[0].set_ylabel('Parameters')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Trainable ratio
    axes[1].bar(df_comparison['config'], df_comparison['trainable_ratio'] * 100, alpha=0.7, color='green')
    axes[1].set_title('Parameter Efficiency by LoRA Configuration')
    axes[1].set_ylabel('Trainable Parameters (%)')
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig(f"{config.output_dir}/lora_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

## 8. Save Trained Model

In [ ]:
# Save the trained model
model_save_path = Path(config.output_dir) / "trained_model"
model_save_path.mkdir(exist_ok=True)

try:
    # Save LoRA weights
    model.save_pretrained(str(model_save_path))
    
    # Save training configuration
    with open(model_save_path / "training_config.json", 'w') as f:
        json.dump(config.to_dict(), f, indent=2)
    
    # Save training history
    if training_history:
        with open(model_save_path / "training_history.json", 'w') as f:
            json.dump(training_history, f, indent=2)
    
    print(f"Model saved to: {model_save_path}")
    print(f"Files created:")
    for file in model_save_path.iterdir():
        print(f"  - {file.name}")
    
except Exception as e:
    print(f"Error saving model: {e}")
    
    # Fallback: save model state dict
    torch.save(model.model.state_dict(), model_save_path / "model_state_dict.pt")
    print(f"Model state dict saved to: {model_save_path / 'model_state_dict.pt'}")

## 9. Training Summary and Insights

In [ ]:
# Generate training summary
print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)

print(f"\nModel Configuration:")
print(f"  Base Model: {config.model_name}")
print(f"  LoRA Rank: 16")
print(f"  LoRA Alpha: 32")
print(f"  Trainable Parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")

print(f"\nTraining Configuration:")
print(f"  Batch Size: {config.batch_size}")
print(f"  Learning Rate: {config.learning_rate}")
print(f"  Epochs: {config.num_train_epochs}")
print(f"  Mixed Precision: {config.bf16 or config.fp16}")

if training_history:
    print(f"\nTraining Results:")
    print(f"  Initial Loss: {training_history[0]['train_loss']:.4f}")
    print(f"  Final Loss: {training_history[-1]['train_loss']:.4f}")
    if len(training_history) > 1:
        improvement = (training_history[0]['train_loss'] - training_history[-1]['train_loss']) / training_history[0]['train_loss'] * 100
        print(f"  Improvement: {improvement:.1f}%")

print(f"\nOutput Files:")
output_path = Path(config.output_dir)
for file in output_path.rglob('*'):
    if file.is_file():
        print(f"  - {file.relative_to(output_path)}")

print(f"\nNext Steps:")
print(f"  1. Evaluate model on larger validation set")
print(f"  2. Try different LoRA configurations")
print(f"  3. Experiment with different learning rates")
print(f"  4. Scale up training with full dataset")
print(f"  5. Deploy model for inference testing")

print("\n" + "=" * 60)

## Conclusion

This notebook demonstrated:

1. **Model Setup**: Initialized CLIP with LoRA for parameter-efficient fine-tuning
2. **Data Pipeline**: Created synthetic multimodal dataset for training
3. **Training Process**: Executed end-to-end training with monitoring
4. **Evaluation**: Tested text and image encoding capabilities
5. **Analysis**: Compared different LoRA configurations
6. **Model Persistence**: Saved trained model for future use

The framework successfully demonstrates production-ready multimodal training capabilities with:
- Parameter-efficient fine-tuning
- Proper data handling
- Training monitoring
- Model evaluation
- Comprehensive logging

For production use, scale up with larger datasets, distributed training, and comprehensive evaluation metrics.